In [1]:
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import math

### Ca Force Data

In [ ]:
# edit path for desired experiment
experiment = ''

# cytosolic Ca channel
_, cyt_data = cv2.imreadmulti(f"ca_force_movies/{experiment}/561_registered.tif", flags=cv2.IMREAD_UNCHANGED)
# traction force channel
_, tract_data = cv2.imreadmulti(f"ca_force_movies/{experiment}/traction_maps.tif", flags=cv2.IMREAD_UNCHANGED)
# mitochondrial Ca channel
_, mit_data = cv2.imreadmulti(f"ca_force_movies/{experiment}/488_registered.tif", flags=cv2.IMREAD_UNCHANGED)

#### Find ROIs based on moving regions in cytosolic calcium data

Detect motion in cyt_data globally (motion masks using absdiff())

In [ ]:
# cyt Ca channel motion masks
cyt_motion = []

for frame_idx in range(1, len(cyt_data)):

    # collect prev and curr frames
    prev_frame = cyt_data[frame_idx-1]
    curr_frame = cyt_data[frame_idx]

    # find differences between frames (pixel absolute difference)
    frame_diff = cv2.absdiff(curr_frame, prev_frame)

    # image processing
    norm = cv2.normalize(frame_diff, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    blur = cv2.GaussianBlur(frame_diff, (17,17), 0)# adjust blur for each experiment
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # append to list
    cyt_motion.append(thresh)

Find centroids based on global motion in cyt_data

In [27]:
# collect rois for each frame in cyt_motion
centroids = []

for mask in cyt_motion:

    # find contours for motion mask
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:

        # get centroids through moments (for tracking)
        M = cv2.moments(contour)

        # avoid divide by 0
        if M["m00"] == 0:
            continue

        cx = int(M['m10']/M['m00'])
        cy = int(M['m01']/M['m00'])

        centroids.append([cx,cy])

Cluster centroids across motion masks

In [32]:
# eps - max distance to be considered in the same cluster (in pixels), min_samples - minimum number of points to form a cluster
all_centroids = np.array(centroids)

# use DBSCAN to cluster centroids found across frames to get ROIs in movement hotspots 
db = DBSCAN(eps=25, min_samples=30).fit(all_centroids) # adjust eps and min_samples to change the size and number of ROIs
labels = db.labels_

# extract centroid cluster IDs, remove -1 for noise
unique_labels = set(labels)
clusters = [all_centroids[labels == l] for l in unique_labels if l != -1]

Collect ROIs and visualize centroids on example cyt Ca frame

In [33]:
# Create an empty mask to draw the grouped ROIs
roi_map = np.copy(cyt_data[0]) # example frame for visualizing
roi_map = cv2.normalize(roi_map, None, 0, 255, cv2.NORM_MINMAX).astype(np.float32)
roi_map = cv2.cvtColor(roi_map, cv2.COLOR_GRAY2BGR)

rois = []

for cluster_points in clusters:
    # get coordiantes for clusters
    x_min, y_min = np.min(cluster_points, axis=0)
    x_max, y_max = np.max(cluster_points, axis=0)
    
    # draw a rectangle around cluster
    cv2.rectangle(roi_map, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
    
    # get new centroids for each ROIS
    centroid_x = int(np.mean(cluster_points[:, 0]))
    centroid_y = int(np.mean(cluster_points[:, 1]))

    # draw centroids on figure
    cv2.circle(roi_map, (centroid_x, centroid_y), 3, (0, 0, 255), -1)

    # collect ROI coords and centroids
    rois.append({
        'dim': (x_min, x_max, y_min, y_max),
        'centroids': (centroid_x, centroid_y)
    })

# add ROI labels to image
for roi_idx, roi in enumerate(rois):

    # grab dimensions for the roi
    xmin, ymin = roi['dim'][0],  roi['dim'][2]

    # display roi label
    text = f"#{roi_idx+1}"

    # position text slightly above ROI
    text_x = xmin
    text_y = max(ymin - 5, 15)                         
    cv2.putText(roi_map, text, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 1, cv2.LINE_AA)

_ = cv2.imwrite(f"{experiment}_mapped_rois.png", roi_map)

#### Find illuminated regions in cyt_data

Get the average intensities from ROIs in the cyt Ca channel

In [17]:
# mean intensities for each ROI across all frames in cytosolic Ca channel TIF
cyt_intensity = []

# masks of each ROI in each frame to pull intensities only from within the cell boundaries
cyt_masks = {}

for frame_idx in range(0, len(cyt_data)):

    curr_frame = cyt_data[frame_idx]
    roi_avg_intensity = []

    for roi_idx, roi in enumerate(rois):

        # grab dimensions for the roi
        xmin, xmax, ymin, ymax = roi['dim'][0], roi['dim'][1], roi['dim'][2], roi['dim'][3]

        # crop frame to ROI
        crop = curr_frame[ymin:ymax, xmin:xmax]

        # create a mask to pull intensities from using contours
        crop_copy = np.copy(crop).astype(np.uint8)
        # image processing
        crop_copy = cv2.GaussianBlur(crop_copy, (3,3), 0)
        _, crop_copy = cv2.threshold(crop_copy, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(crop_copy, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        mask = np.zeros(crop.shape[:2], dtype=np.uint8)
        cv2.drawContours(mask, contours, -1, 255, thickness=-1)

        # add masks to dict cyt_masks; key consists of frame index, roi index for each mask
        cyt_masks[(frame_idx, roi_idx)] = mask

        # get mean pixel intensity for cell in that ROI
        avg_intensity = cv2.mean(crop, mask=mask)

        roi_avg_intensity.append(avg_intensity[0])

    cyt_intensity.append(roi_avg_intensity)

# average intensities down columns to get the avg intensity per ROI across frames
cyt_avg_intensity = np.mean(cyt_intensity, axis=0)

Display the average cytosolic Ca intensity per ROI

In [18]:
# use example frame to show avg cyt intensities per ROI
cyt_map = cv2.normalize(cyt_data[0], None, 0, 255, cv2.NORM_MINMAX).astype(np.float32)
cyt_map = cv2.cvtColor(cyt_map, cv2.COLOR_GRAY2BGR)

for roi, intensity in zip(rois, cyt_avg_intensity):

    # get ROI coords
    xmin, xmax, ymin, ymax = roi['dim']
    cx,cy = roi['centroids']

    # draw ROI
    cv2.rectangle(cyt_map, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)

    # draw centroid
    cv2.circle(cyt_map, (cx, cy), 3, (0, 0, 255), -1)

    # display average intensity text
    text = f"#{rois.index(roi)+1}: {intensity:.1f}"

    # position text slightly above ROI
    text_x = xmin
    text_y = max(ymin - 5, 15)

    cv2.putText(cyt_map, text, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1, cv2.LINE_AA)

_ = cv2.imwrite(f"{experiment}_avg_cyt_int.png", cyt_map)

#### Find illuminated regions in tract_data

Get the average intensities from ROIs in the traction force channel

In [19]:
# mean intensities for each ROI across all frames in traction force channel TIF
tract_intensity = []

for frame_idx in range(0, len(tract_data)):

    curr_frame = tract_data[frame_idx]
    roi_avg_intensity = []

    for roi in rois:
        # grab dimensions for the roi
        xmin, xmax, ymin, ymax = roi['dim'][0], roi['dim'][1], roi['dim'][2], roi['dim'][3]

        # crop frame to ROI
        crop = curr_frame[ymin:ymax, xmin:xmax]

        # get mean pixel intensity for cell in that ROI
        avg_intensity = cv2.mean(crop)

        roi_avg_intensity.append(avg_intensity[0])

    tract_intensity.append(roi_avg_intensity)

# average intensities down columns to get the avg intensity per ROI across frames
tract_avg_intensity = np.mean(tract_intensity, axis=0)

In [20]:
# use example frame to show avg traction force intensities per ROI on cyt data
tract_map = cv2.normalize(cyt_data[0], None, 0, 255, cv2.NORM_MINMAX).astype(np.float32)
tract_map = cv2.cvtColor(tract_map, cv2.COLOR_GRAY2BGR)

for roi, intensity in zip(rois, tract_avg_intensity):

    # get ROI coords
    xmin, xmax, ymin, ymax = roi['dim']
    cx,cy = roi['centroids']

    # draw ROI
    cv2.rectangle(tract_map, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)

    # draw centroid
    cv2.circle(tract_map, (cx, cy), 3, (0, 0, 255), -1)

    # display average intensity text
    text = f"#{rois.index(roi)+1}: {intensity:.1f}"

    # position text slightly above ROI
    text_x = xmin
    text_y = max(ymin - 5, 15)

    cv2.putText(tract_map, text, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1, cv2.LINE_AA)
  
_ = cv2.imwrite(f"{experiment}_avg_tract_int.png", tract_map)

#### Find illuminated regions in mit_data

Get the average intensities from ROIs in the mit Ca channel

In [21]:
# mean intensities for each ROI across all frames in traction force channel TIF
mit_intensity = []

for frame_idx in range(0, len(mit_data)):

    curr_frame = mit_data[frame_idx]
    roi_avg_intensity = []

    for roi_idx, roi in enumerate(rois):
        # grab dimensions for the roi
        xmin, xmax, ymin, ymax = roi['dim'][0], roi['dim'][1], roi['dim'][2], roi['dim'][3]

        # crop frame to ROI
        crop = curr_frame[ymin:ymax, xmin:xmax]

        # get mask for current frame and ROI
        mask = cyt_masks[frame_idx, roi_idx]

        # get mean pixel intensity for cell in that ROI
        avg_intensity = cv2.mean(crop, mask=mask)

        roi_avg_intensity.append(avg_intensity[0])

    mit_intensity.append(roi_avg_intensity)

# average intensities down columns to get the avg intensity per ROI across frames
mit_avg_intensity = np.mean(mit_intensity, axis=0)

In [22]:
# use example frame to show avg traction force intensities per ROI on cyt data
mit_map = cv2.normalize(cyt_data[0], None, 0, 255, cv2.NORM_MINMAX).astype(np.float32)
mit_map = cv2.cvtColor(mit_map, cv2.COLOR_GRAY2BGR)

for roi, intensity in zip(rois, mit_avg_intensity):

    # get ROI coords
    xmin, xmax, ymin, ymax = roi['dim']
    cx,cy = roi['centroids']

    # draw ROI
    cv2.rectangle(mit_map, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)

    # draw centroid
    cv2.circle(mit_map, (cx, cy), 3, (0, 0, 255), -1)

    # display average intensity text
    text = f"#{rois.index(roi)+1}: {intensity:.1f}"

    # position text slightly above ROI
    text_x = xmin
    text_y = max(ymin - 5, 15)

    cv2.putText(mit_map, text, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1, cv2.LINE_AA)

_ = cv2.imwrite(f"{experiment}_avg_mit_int.png", mit_map)

#### Other Figures

In [ ]:
# Cyt Ca and traction force pixel intensities over time per ROI
n_frames, n_rois = np.array(cyt_intensity).shape
time = np.arange(n_frames)

# number of cols and rows
ncols = 4
nrows = math.ceil(n_rois / ncols)

# create faceted subplots
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4*ncols, 3*nrows),
    sharex=True,
    sharey=True,
    constrained_layout=True
)

axes = np.ravel(axes)

# plot for each ROI
for roi in range(n_rois):

    ax = axes[roi]

    ax.plot(time, np.array(cyt_intensity)[:,roi], lw=1.5, label='Cytosolic Ca', color='blue')
    ax.set_title(f"ROI #{roi+1}", fontsize=12)

# remove unused axes
for ax in axes[n_rois:]:
    fig.delaxes(ax)

# plot labels
fig.suptitle("Cyt Ca Avg. Intensities per Frame", fontsize=18)
fig.supxlabel("Frame", fontsize=16)
fig.supylabel("Avg Pixel Intensity", fontsize=16)

# legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', ncol=2, fontsize=16, frameon=True)

plt.savefig(f'{experiment}_cyt_facet.png')
plt.show()

In [ ]:
# create faceted subplots 
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4*ncols, 3*nrows),
    sharex=True,
    sharey=True,
    constrained_layout=True
)

axes = np.ravel(axes)

# plot for each ROI
for roi in range(n_rois):

    ax = axes[roi]

    ax.plot(time, np.array(tract_intensity)[:,roi], lw=1.5, label='Traction Force', color='orange')
    ax.set_title(f"ROI #{roi+1}", fontsize=12)

# remove unused axes
for ax in axes[n_rois:]:
    fig.delaxes(ax)

# plot labels
fig.suptitle("Traction Force Avg. Intensities per Frame", fontsize=18)
fig.supxlabel("Frame", fontsize=16)
fig.supylabel("Avg Pixel Intensity", fontsize=16)

# legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right', ncol=2, fontsize=16, frameon=True)

plt.savefig(f'{experiment}_tract_facet.png')
plt.show()